# llcAPI_test

## See ledger/llcAPI.py

In [1]:
# Load bookkeeping services
import os, sys
from pathlib import Path
if len([p for p in sys.path if 'Ledger' in p]) == 0:
    sys.path.append(os.path.join(os.getcwd(), 'Ledger'))

## TEST : Load profile 

In [2]:
from ledger.LLC import LLC
import datetime
from IPython.display import display, Markdown

# Load LLC and its profile
top = os.path.join(Path.home(), 'GDrive/Family/Assets-Hobby/RealEstateInvestments/LLC-WB-Group')
llc = LLC('WBGroupLLC',debug=True, top=top)

# Summarize LLC
dtReport = datetime.datetime.now().strftime('%Y.%m.%d')
display(Markdown(f"### Profile - {dtReport}"))
display(Markdown(f"- **LLC Name: {llc.objName}**"))
display(Markdown(f"- **Year: {llc.yr}**"))

llc:LLC init load _Profile
LLC llcProfile FN /Users/frankrojas/GDrive/Family/Assets-Hobby/RealEstateInvestments/LLC-WB-Group/llcProfile_WBGroupLLC.json
Profile loaded /Users/frankrojas/GDrive/Family/Assets-Hobby/RealEstateInvestments/LLC-WB-Group/llcProfile_WBGroupLLC.json
llc:LLC LLC Init Done


### Profile - 2026.03.14

- **LLC Name: WBGroupLLC**

- **Year: 2025**

## Test: Load Bank csv 

In [3]:
from ledger.llcBank import llcBank

bk = llcBank(llc, debug=True)
bk.fetch()

llcBank llcBank Init Done
llcBank llcBank Init Done
llcBank dwnLdCSV: importBankCSV csvBN: WBGroupLLC_WF_20251231.csv
llcBank CSV Loaded /Users/frankrojas/GDrive/Family/Assets-Hobby/RealEstateInvestments/LLC-WB-Group/pages/AccountingData/2025/BankStmts/WBGroupLLC_WF_20251231.csv
llcAssets llcAssets Init Done
llc:llcAssets llcAssets Init Done
llcAssets ledgerObject.FN: /Users/frankrojas/GDrive/Family/Assets-Hobby/RealEstateInvestments/LLC-WB-Group/pages/AccountingData/Accts/llcAssets_WBGroupLLC.json
llcAssets ledgerObject.FN: /Users/frankrojas/GDrive/Family/Assets-Hobby/RealEstateInvestments/LLC-WB-Group/pages/AccountingData/Accts/llcAssets_WBGroupLLC.json
ledgerClassify ledgerClassify Init Done


In [4]:
bk.df

,dt,amt,C2,CheckNo,desc,TransType,Acct,AcctSub,TDesc
0,12/29/2025,-177.00,*,NaN,Cash eWithdrawal in Branch 12/29/2025 13:47 PM...,Exp,Acct.Asset.Purchase,p20251229-RV1,Purchase Rental RV
1,12/29/2025,177.00,*,NaN,eDeposit in Branch 12/29/25 03:48:15 PM 14650 ...,Rev,Acct.Cash.Investment,o20250801_1,Initial Investment by member
2,12/26/2025,-135.80,*,NaN,ALLSTATE IND CO INS PYMT DEC024 00000043853221...,Exp,Acct.Cash.Util,Ins_Home,Pay Monthly Util
3,12/18/2025,-50.00,*,NaN,BILL PAY Water-COMWSC RECURRING 38 ON 12-18,Exp,Acct.Cash.Util,Water,Pay Monthly Util
4,12/10/2025,-129.16,*,NaN,BUSINESS TO BUSINESS ACH Pedernales_Elec ELEC_...,Exp,Acct.Cash.Util,Elec,Pay Monthly Util
5,12/01/2025,1500.00,*,NaN,ZELLE FROM NICOLA ROJAS ON 12/01 REF # BBT3528...,Rev,Acct.Cash.Income,c20251001-1,Rental Income
6,11/18/2025,-50.00,*,NaN,BILL PAY Water-COMWSC RECURRING 38 ON 11-18,Exp,Acct.Cash.Util,Water,Pay Monthly Util
7,11/17/2025,-2.00,*,NaN,PURCHASE AUTHORIZED ON 11/15 HAYS CO TX WIMBER...,Exp,Acct.Cash.Expense,hays,Expense: 11/15 hays co tx wimber fort worth tx...
8,11/17/2025,-30.00,*,NaN,PURCHASE AUTHORIZED ON 11/15 HAYS CO TX WIMBER...,Exp,Acct.Cash.Expense,hays,Expense: 11/15 hays co tx wimber san marcos tx...
9,11/17/2025,-19.47,*,NaN,PURCHASE AUTHORIZED ON 11/15 AMAZON MKTPL*B80W...,Exp,Acct.Cash.Expense,amazon,Expense: 11/15 amazon mktpl*b80w8 amzn.com/bil...


## Test: Tranaction Classification

In [6]:
from ledger.llcAssets import llcAssets           
a = llcAssets(llc)
a.fetch()
r = bk.df.iloc[0]
a._matchBk(r)

s  = bk.df.apply(lambda r : a._matchBk(r), axis=1)
[r for r in s if r is not None]

[('Acct.Asset.Purchase', 'p20251229-RV1', 'Purchase Rental RV'),
 ('Acct.Cash.Investment', 'o20250801_1', 'Initial Investment by member'),
 ('Acct.Asset.Purchase',
  'p20250826-805HMD',
  'Purchase Property: 805 High Mesa'),
 ('Acct.Cash.Investment', 'o20250801_1', 'Open Bank Acct Investment'),
 ('Acct.Cash.Investment', 'o20250801_1', 'Initial Investment by member')]

## Test ledget Iterator

In [15]:
bk.df[bk.df.Acct.str.contains('Investment')]

,dt,amt,C2,CheckNo,desc,TransType,Acct,AcctSub,TDesc
1,12/29/2025,177.0,*,NaN,eDeposit in Branch 12/29/25 03:48:15 PM 14650 ...,Rev,Acct.Cash.Investment,o20250801_1,Initial Investment by member
52,08/20/2025,50.0,*,NaN,WFB OPENING DEPOSIT FROM CARD XXXXXXXXXXXX1980...,Rev,Acct.Cash.Investment,o20250801_1,Open Bank Acct Investment
53,08/20/2025,219000.0,*,NaN,WT FED#02M03 NATIONAL FINANCIAL /ORG=FRANCIS X...,Rev,Acct.Cash.Investment,o20250801_1,Initial Investment by member


In [11]:
# filter transactions X assets
import pandas as pd
aDF = llc.aObj.df
iDF = pd.merge(aDF, bk.df, 
         left_on=['dt','amt'], 
         right_on=['dt', 'amt'],
         how='inner')
iDF[['acct', 'amt', 'desc_x', 'AcctSub']]
bal = ('Balance_Acct.Cash')
iDF.loc[bal] = [llc.objName, float(iDF.amt.sum()),'LLC']
llcBalInvestment = iDF.amt.iloc[-1]
iDF

,acct,amt,desc_x,AcctSub
0,Acct.Cash.Investment,50.00,Open Bank Acct Investment,o20250801_1
1,Acct.Asset.Purchase,-213936.95,Purchase Property: 805 High Mesa,p20250826-805HMD
2,Acct.Cash.Investment,219000.00,Initial Investment by member,o20250801_1
3,Acct.Asset.Purchase,-177.00,Purchase Rental RV,p20251229-RV1
4,Acct.Cash.Investment,177.00,Initial Investment by member,o20250801_1


In [ ]:
bk.df